# VulnTriage-LLM: QLoRA Fine-Tuning

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mon Apr 27 19:48:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             75W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'datasets', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'scikit-learn'], check=True)
subprocess.run(['pip', 'install', '-q', 'torchao>=0.16.0'], check=True)
print('Done.')

Done.


In [4]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('HuggingFace login successful.')

HuggingFace login successful.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

DATA_DIR   = '/content/drive/MyDrive/VulnTriage-LLM'
OUTPUT_DIR = '/content/drive/MyDrive/VulnTriage-LLM/llama-vuln-qlora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df = pd.read_json(f'{DATA_DIR}/train.json')
val_df   = pd.read_json(f'{DATA_DIR}/val.json')
test_df  = pd.read_json(f'{DATA_DIR}/test.json')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(train_df['severity'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 52052 | Val: 24976 | Test: 67823
severity
HIGH        22118
MEDIUM      21322
CRITICAL     7246
LOW          1366
Name: count, dtype: int64


In [6]:
import torch

vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_4BIT = vram_gb < 30

BATCH_SIZE = 16 if not USE_4BIT else 4
GRAD_ACCUM = 2  if not USE_4BIT else 4
LORA_R     = 32 if not USE_4BIT else 8

print(f'VRAM: {vram_gb:.1f} GB')
print(f'Strategy: {"Full bfloat16" if not USE_4BIT else "QLoRA 4-bit"}')
print(f'Batch size: {BATCH_SIZE} | Grad accum: {GRAD_ACCUM} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'LoRA rank: {LORA_R}')

VRAM: 42.4 GB
Strategy: Full bfloat16
Batch size: 16 | Grad accum: 2 | Effective batch: 32
LoRA rank: 32


In [7]:
from datasets import Dataset

SYSTEM = (
    'You are a cybersecurity expert. Given a CVE vulnerability description, '
    'predict its CVSS v3.1 severity. Reply with exactly one word: '
    'CRITICAL, HIGH, MEDIUM, or LOW.'
)

def format_sample(row):
    text = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        + SYSTEM + '<|eot_id|>'
        + '<|start_header_id|>user<|end_header_id|>\n'
        + 'Vulnerability: ' + row['description'] + '<|eot_id|>'
        + '<|start_header_id|>assistant<|end_header_id|>\n'
        + row['severity'] + '<|eot_id|>'
    )
    return {'text': text}

train_data = Dataset.from_list([format_sample(r) for _, r in train_df.iterrows()])
val_data   = Dataset.from_list([format_sample(r) for _, r in val_df.iterrows()])

print(f'Train: {len(train_data)} | Val: {len(val_data)}')
print('\nExample:')
print(train_data[0]['text'][:300])

Train: 52052 | Val: 24976

Example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a cybersecurity expert. Given a CVE vulnerability description, predict its CVSS v3.1 severity. Reply with exactly one word: CRITICAL, HIGH, MEDIUM, or LOW.<|eot_id|><|start_header_id|>user<|end_header_id|>
Vulnerability: An informat


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID   = 'meta-llama/Llama-3.1-8B'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded.')

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
print('Model loaded.')

`torch_dtype` is deprecated! Use `dtype` instead!


Tokenizer loaded.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded.


In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


In [13]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    bf16=True,
    optim='paged_adamw_8bit',
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    warmup_steps=100,
    lr_scheduler_type='cosine',
    report_to='none',
    seed=42,
    max_length=MAX_LENGTH,
    packing=False,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
)

print(f'Steps per epoch: {len(train_data) // (BATCH_SIZE * GRAD_ACCUM)}')
print('Starting training...')
trainer.train()

Adding EOS to train dataset:   0%|          | 0/52052 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52052 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/24976 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/24976 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Steps per epoch: 1626
Starting training...


Step,Training Loss,Validation Loss
500,1.213919,1.448265
1000,1.190064,1.446616
1500,1.161661,1.438611
2000,1.002349,1.466934
2500,1.010467,1.458661
3000,0.977516,1.455900
3500,0.810999,1.526634
4000,0.820447,1.533285
4500,0.778081,1.535173


TrainOutput(global_step=4881, training_loss=1.0108355155027295, metrics={'train_runtime': 19087.3307, 'train_samples_per_second': 8.181, 'train_steps_per_second': 0.256, 'total_flos': 1.6455937854923735e+18, 'train_loss': 1.0108355155027295})

In [14]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

Model saved to /content/drive/MyDrive/VulnTriage-LLM/llama-vuln-qlora


In [15]:
from peft import PeftModel
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
import pandas as pd
import torch

print('Loading fine-tuned model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
ft_model.eval()
print('Model loaded.')

VALID_LABELS = {'CRITICAL', 'HIGH', 'MEDIUM', 'LOW'}

def predict(description):
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        + SYSTEM + '<|eot_id|>'
        + '<|start_header_id|>user<|end_header_id|>\n'
        + 'Vulnerability: ' + description + '<|eot_id|>'
        + '<|start_header_id|>assistant<|end_header_id|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    raw = decoded.strip().upper()
    for label in VALID_LABELS:
        if label in raw:
            return label
    return 'UNKNOWN'

N_PER_CLASS = 250
sampled = pd.concat([
    resample(test_df[test_df['severity'] == label], n_samples=N_PER_CLASS, random_state=42, replace=False)
    for label in VALID_LABELS
]).reset_index(drop=True)

print(f'Evaluating on {len(sampled)} samples...')
predictions = []
for i, row in sampled.iterrows():
    pred = predict(row['description'])
    predictions.append(pred)
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(sampled)}')

sampled['predicted'] = predictions
sampled.to_json(f'{DATA_DIR}/finetuned_results.json', orient='records', indent=2)

clean = sampled[sampled['predicted'] != 'UNKNOWN']
print(f'Parseable: {len(clean)}/{len(sampled)}')
print('\n--- Classification Report ---')
print(classification_report(clean['severity'], clean['predicted'], labels=['CRITICAL', 'HIGH', 'MEDIUM', 'LOW'], digits=4))
macro_f1 = f1_score(clean['severity'], clean['predicted'], labels=['CRITICAL', 'HIGH', 'MEDIUM', 'LOW'], average='macro')
print(f'Macro F1: {macro_f1:.4f}')
print('Results saved to finetuned_results.json')

Loading fine-tuned model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
Evaluating on 1000 samples...
  100/1000
  200/1000
  300/1000
  400/1000
  500/1000
  600/1000
  700/1000
  800/1000
  900/1000
  1000/1000
Parseable: 999/1000

--- Classification Report ---
              precision    recall  f1-score   support

    CRITICAL     0.6561    0.5800    0.6157       250
        HIGH     0.4465    0.5840    0.5061       250
      MEDIUM     0.3841    0.6787    0.4906       249
         LOW     0.7273    0.0320    0.0613       250

    accuracy                         0.4685       999
   macro avg     0.5535    0.4687    0.4184       999
weighted avg     0.5537    0.4685    0.4183       999

Macro F1: 0.4184
Results saved to finetuned_results.json
